import libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier , RandomForestRegressor
from sklearn.metrics import mean_squared_error
from datetime import datetime , timedelta
import pytz

In [ ]:
API_KEY = '12e527074275c78bf702f0207a7c0a93'
BASE_URL = 'https://api.openweathermap.org/data/3.0/'

FETCHING CURRENT DATA

In [ ]:
def get_current_weather(city):
  url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric" # Changed endpoint to 2.5/weather
  response = requests.get(url)
  data = response.json()
  return {
      'city' : data['name'],
      'current_temp' : round(data['main']['temp']),
      'feels_like' : round(data['main']['feels_like']),
      'temp_min' : round(data['main']['temp_min']),
      'temp_max' : round(data['main']['temp_max']),
      'humidity' : round(data['main']['humidity']),
      'discription' : data['weather'][0]['description'],
      'country' : data['sys']['country'],
      'wind_gust_dir' : data['wind']['deg'],
      'pressure' : data['main']['pressure'],
      'Wind_Gust_Speed' : data['wind']['speed']
  }

read previous data

In [ ]:
def read_historical_data(filename):
    df = pd.read_csv(filename)
    df = df.dropna()
    df = df.drop_duplicates()
    return df

DATA PREPRATION


In [ ]:
def prepare_data(data):
  le = LabelEncoder()
  data['WindGustDir'] = le.fit_transform(data['WindGustDir'])
  data['RainTomorrow'] = le.fit_transform(data['RainTomorrow'])

  X = data[['MinTemp', 'MaxTemp', 'WindGustDir', 'WindGustSpeed', 'Humidity', 'Pressure', 'Temp']]
  y = data['RainTomorrow']
  return X, y, le

train model

In [ ]:
def train_rain_model(X, y):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
  model = RandomForestClassifier(n_estimators=100, random_state=42)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  print("mean squared error for rain model")
  print(mean_squared_error(y_test, y_pred))
  return model

prepare regression data


In [ ]:
def prepare_regression_data(data, feature):
  X, y = [], []
  for i in range(len(data) - 1 ):
    X.append(data[feature].iloc[i])
    y.append(data[feature].iloc[i + 1])

  X = np.array(X).reshape(-1, 1)
  y = np.array(y)
  return X, y


train regression model


In [ ]:
def train_regression_model(X, y):
  model = RandomForestRegressor(n_estimators=100, random_state=42)
  model.fit(X, y)
  return model

predict future

In [ ]:
def predict_future(model, current_value):
  predctions = [current_value]
  for i in range(5):
    # Reshape to (1, -1) to provide a 2D array for a single sample
    next_value = model.predict(np.array([predctions[-1]]).reshape(1, -1))
    predctions.append(next_value[0])

  return predctions[1:]

weather analysis

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

# Assuming the uploaded file is named 'weather.csv'
# If your file has a different name, please adjust the path in the next step accordingly.

Saving weather.csv to weather (1).csv
User uploaded file "weather (1).csv" with length 12536 bytes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def weather_view():
  city = input("enter city name")
  current_weather = get_current_weather(city)
  historical_data = read_historical_data('/content/weather.csv')
  X, y, le = prepare_data(historical_data)
  rain_model = train_rain_model(X, y)

  # map wind direction to compass points
  wind_deg = current_weather['wind_gust_dir'] % 360

  compass_points = [
    ("N", 0, 11.25), ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
    ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75),
    ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
    ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75),
    ("W", 258.75, 281.25), ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
    ("NNW", 326.25, 348.75)
]

  compass_direction = next((point for point, start, end in compass_points if start <= wind_deg < end), None) if wind_deg is not None else None


  compass_direction_encoded = le.transform([compass_direction])[0] if compass_direction in le.classes_ else -1


  current_data = {
      'MinTemp' : current_weather['temp_min'],
      'MaxTemp' : current_weather['temp_max'],
      'WindGustDir': compass_direction_encoded,
      'WindGustSpeed' : current_weather['Wind_Gust_Speed'],
      'Humidity' : current_weather['humidity'],
      'Pressure' : current_weather['pressure'],
      'Temp' : current_weather['current_temp'],

  }
  current_df = pd.DataFrame([current_data])
  rain_prediction = rain_model.predict(current_df)[0]

  # Fixed 'temp' to 'Temp' and 'humidity' to 'Humidity' to match DataFrame columns
  X_temp, y_temp = prepare_regression_data(historical_data, 'Temp')
  X_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')
  temp_model = train_regression_model(X_temp, y_temp)
  hum_model = train_regression_model(X_hum, y_hum)

  future_temp = predict_future(temp_model, current_weather['temp_min'])
  future_hum = predict_future(hum_model, current_weather['humidity'])

  timezone = pytz.timezone('Asia/Kolkata')
  current_time = datetime.now(timezone)
  next_hour = current_time + timedelta(hours=1)
  next_hour = next_hour.replace(minute=0, second=0, microsecond=0)

  future_times = [(next_hour + timedelta(hours=i)).strftime("%H:00") for i in range(5)]

  print(f"City : {city}, {current_weather['city']}")
  print(f"Current Temperature : {current_weather['current_temp']}")
  print(f"Feels Like : {current_weather['feels_like']}")
  print(f"min Temperature : {current_weather['temp_min']}")
  print(f"max Temperature : {current_weather['temp_max']}")
  print(f"Humidity : {current_weather['humidity']}")
  print(f"Weather Description : {current_weather['discription']}")
  print(f"Rain prediction : {'Yes' if rain_prediction == 1 else 'No'}")


  print("\nFuture Temperatures Predictions :")
  for time, temp in zip(future_times, future_temp):
    print(f"{time}: {round(temp, 1)}")

  print("\nFuture Humidity Predictions :")
  for time, hum in zip(future_times, future_hum):
    print(f"{time}: {round(hum, 1)}")

weather_view()

enter city namedelhi
mean squared error for rain model
0.1506849315068493
City : delhi, Delhi
Current Temperature : 20
Feels Like : 20
min Temperature : 20
max Temperature : 20
Humidity : 83
Weather Description : mist
Rain prediction : Yes

Future Temperatures Predictions :
01:00: 22.1
02:00: 23.3
03:00: 19.0
04:00: 24.5
05:00: 24.5

Future Humidity Predictions :
01:00: 54.1
02:00: 58.1
03:00: 54.2
04:00: 58.1
05:00: 54.2
